# ModernFinBERT v2: Entity-Aware Train with Unsloth (8192 Context)

Train ModernBERT-base with LoRA using Unsloth for memory-efficient full 8192-token training on T4 GPU.
- `neoyipeng/modernfinbert-training-v2` (43K rows, short-context: headlines, tweets, analyst reports, earnings excerpts)
- `neoyipeng/modernfinbert-training-v2-long` (36K rows, long-context: earnings call transcripts, SEC 10-K MD&A)

Entity-aware input: uses sentence pair format `[CLS] entity [SEP] text [SEP]` so the model
learns entity-targeted sentiment rather than overall document sentiment.

Push best model to HuggingFace. Benchmarking done separately in `02_benchmark_inference.ipynb`.

In [ ]:
%%capture
import os, re, torch
v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer scikit-learn
!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
%env UNSLOTH_DISABLE_FAST_GENERATION=1

import torch
import numpy as np
from unsloth import FastModel, is_bfloat16_supported
from datasets import load_dataset, concatenate_datasets
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 3407
NUM_CLASSES = 3
LABEL_NAMES = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
LABEL_TO_ID = {"NEGATIVE": 0, "NEUTRAL": 1, "POSITIVE": 2}
ID_TO_LABEL = {0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"}
MAX_LENGTH = 8192
MODEL_NAME = "unsloth/ModernBERT-base"

cap = torch.cuda.get_device_capability()
assert cap[0] >= 7, f"GPU sm_{cap[0]}{cap[1]} not supported — need sm_70+ (T4/V100/A100)"
print(f"GPU: {torch.cuda.get_device_name()} (sm_{cap[0]}{cap[1]})")
print(f"Max tokens: {MAX_LENGTH}")

## 1. Load & Combine Training Data

In [ ]:
ds_short = load_dataset("neoyipeng/modernfinbert-training-v2")
ds_long = load_dataset("neoyipeng/modernfinbert-training-v2-long")

print("Short-context:", ds_short)
print("\nLong-context:", ds_long)

ds = {}
for split in ["train", "validation", "test"]:
    short_split = ds_short[split].select_columns(["text", "entity", "entity_sentiment"])
    long_split = ds_long[split].select_columns(["text", "entity", "entity_sentiment"])
    ds[split] = concatenate_datasets([short_split, long_split]).shuffle(seed=SEED)

print(f"\nCombined:")
for split in ds:
    print(f"  {split}: {len(ds[split])} rows")

# Map string labels to integers (using entity_sentiment as training target)
def map_labels(examples):
    examples["label"] = [LABEL_TO_ID[l] for l in examples["entity_sentiment"]]
    return examples

for split in ds:
    ds[split] = ds[split].map(map_labels, batched=True)
    ds[split] = ds[split].remove_columns(["entity_sentiment"])

labels = ds["train"]["label"]
print(f"\nLabel distribution (train, entity_sentiment):")
for i, name in enumerate(LABEL_NAMES):
    print(f"  {name}: {labels.count(i)}")

# Entity coverage
entities = ds["train"]["entity"]
n_with_entity = sum(1 for e in entities if e not in ("NONE", "MARKET", "", None))
print(f"\nEntity coverage: {n_with_entity}/{len(entities)} ({100*n_with_entity/len(entities):.1f}%) rows have a named entity")

## 2. Model + LoRA Setup

In [ ]:
model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    auto_model=AutoModelForSequenceClassification,
    max_seq_length=MAX_LENGTH,
    dtype=None,
    num_labels=NUM_CLASSES,
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
    load_in_4bit=False,
)

In [ ]:
model = FastModel.get_peft_model(
    model,
    r=16,
    target_modules=["Wqkv", "out_proj", "Wi", "Wo"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
    task_type="SEQ_CLS",
)

In [ ]:
# Entity-aware tokenization: sentence pair format [CLS] entity [SEP] text [SEP]
# For rows without a meaningful entity (NONE/MARKET), pass empty string as sentence A.
def tokenize_function(examples):
    entities = [e if e not in ("NONE", "MARKET", "", None) else "" for e in examples["entity"]]
    return tokenizer(entities, examples["text"], truncation=True, max_length=MAX_LENGTH)

tokenized = {}
for split in ds:
    tokenized[split] = ds[split].map(tokenize_function, batched=True, remove_columns=["text", "entity"])
    tokenized[split] = tokenized[split].rename_column("label", "labels")
    print(f"{split}: {len(tokenized[split])} rows tokenized")

## 3. Training

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
    }

# Memory test results (NB03 on T4 16GB):
#   seq_len  8192 at batch=8 → 5.8 GB peak (safe margin)
#   seq_len  8192 at batch=16 → 10.8 GB (OOM with Trainer overhead)
# 1 epoch to fit within Kaggle 12h limit (~5.5h estimated).
# Effective batch = 8 * 4 = 32.

trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics,
    args=TrainingArguments(
        output_dir="./modernfinbert-v2-output",
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        weight_decay=0.001,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        optim="adamw_8bit",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        seed=SEED,
        group_by_length=True,
        save_strategy="epoch",
        eval_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="none",
    ),
)

trainer.train()

In [ ]:
test_results = trainer.evaluate(tokenized["test"])
print(f"Test: accuracy={test_results['eval_accuracy']:.4f}, macro_f1={test_results['eval_macro_f1']:.4f}")

print(f"\nTraining log:")
for entry in trainer.state.log_history:
    if "eval_loss" in entry:
        print(f"  Epoch {entry.get('epoch', '?')}: loss={entry['eval_loss']:.4f}, "
              f"acc={entry.get('eval_accuracy', 0):.4f}, f1={entry.get('eval_macro_f1', 0):.4f}")

## 4. Push Model to HuggingFace

In [ ]:
merged = model.merge_and_unload()
merged.push_to_hub("neoyipeng/ModernFinBERT-v2", private=False)
tokenizer.push_to_hub("neoyipeng/ModernFinBERT-v2", private=False)
print("Model pushed to neoyipeng/ModernFinBERT-v2")